# ReplayGym Stage B: Qwen3-1.7B expansion policy (DPO → GRPO) on Colab A100

**Before running:** (1) upload `replaygym_colab_bundle.zip` to `MyDrive/replaygym/`; (2) in the 🔑 Secrets panel add
`AWS_BEARER_TOKEN_BEDROCK` (for GRPO / eval rewards via Bedrock) and `DEEPRESEARCHGYM_API_KEY` (not needed for training), with notebook access on.
Runtime → Change runtime type → **A100 GPU**. Then Run all. Everything is saved to `MyDrive/replaygym/out/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib
D = pathlib.Path('/content/drive/MyDrive/replaygym'); OUT = D/'out'; OUT.mkdir(parents=True, exist_ok=True)
!rm -rf /content/work && mkdir -p /content/work && cd /content/work && unzip -q {D}/replaygym_colab_bundle.zip
%cd /content/work/deepresearchagent

In [ ]:
%pip install -q -e . -e ../gpt-researcher peft gymnasium sentence-transformers
import os
from google.colab import userdata
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = userdata.get('AWS_BEARER_TOKEN_BEDROCK')
# every OpenAI-SDK call (report writer, judge) goes to Bedrock, same as training on the laptop
os.environ.update({
  'OPENAI_API_KEY': os.environ['AWS_BEARER_TOKEN_BEDROCK'],
  'OPENAI_BASE_URL': 'https://bedrock-mantle.us-east-1.api.aws/v1',
  'REPLAY_AGENT_CONFIG': 'configs/agents/gpt_researcher_gym_bedrock.yaml',
  'REPLAY_JUDGE_MODEL': 'qwen.qwen3-32b', 'REPLAY_EMBED': 'local', 'REPLAY_TAG': 'bedrock',
  'REPLAY_SYNTH_CONC': '1', 'REPLAY_Q_WORKERS': '3',
  'LLM_KWARGS': '{"timeout": 300, "max_retries": 2}', 'TOKENIZERS_PARALLELISM': 'false'})
!python -m pytest -q tests/test_replay.py 2>&1 | tail -1

## 1) DPO warm start: offline, no API calls
Pairs come from the pairwise judge memo: "sub-tree A beat sub-tree B for this question". Each sub-tree maps to exactly one decision sequence, so the preference is over exact decision sequences.

In [ ]:
!python scripts/replaygym_lm_train.py dpo --model Qwen/Qwen3-1.7B --epochs 2 --lr 2e-5 --beta 0.1 --margin 0.25
!cp -r runs/replaygym/bedrock/lm_dpo {OUT}/

## 2) GRPO from the DPO adapter (pairwise tournament reward via Bedrock; memoized)

In [ ]:
!python scripts/replaygym_lm_train.py grpo --model Qwen/Qwen3-1.7B --init runs/replaygym/bedrock/lm_dpo --iters 20 --batch 4 --group 8 --lr 1e-5
!cp -r runs/replaygym/bedrock/lm_grpo runs/replaygym/bedrock/memo runs/replaygym/bedrock/pairs {OUT}/

## 3) Validation (10 held-out val questions, replay): LM policy vs full tree / depth-1 / depth-2

In [ ]:
!python scripts/replaygym_lm_train.py eval --model Qwen/Qwen3-1.7B --init runs/replaygym/bedrock/lm_grpo --split val | tee {OUT}/eval_val_lm.txt